# Experiment 4: Multi-Task Validation & Quantization
**Tasks:** Seg, Pose, PTQ, QAT, INT8 export
**Expected time:** ~6-8 hours total

In [ ]:
!rm -rf /content/tinyYOLO 2>/dev/null
!git clone https://github.com/ShMazumder/tinyYOLO.git /content/tinyYOLO
%cd /content/tinyYOLO
!pip install -e . -q && pip install tqdm -q
import torch, os, json, glob
from pathlib import Path
print(f'GPU: {torch.cuda.get_device_name(0)}')

WARMUP = 3

## Part A: Segmentation

In [ ]:
get_ipython().system('python scripts/train.py --task seg --variant quantized --data coco128.yaml --imgsz 416 --epochs 200 --seed 42 --warmup 3 --name seg-q-416-seed42')
get_ipython().system('python scripts/train.py --task seg --variant standard --data coco128.yaml --imgsz 416 --epochs 200 --seed 42 --warmup 3 --name seg-std-416-seed42')
print('✅ Segmentation complete!')

## Part B: Pose Estimation

In [ ]:
get_ipython().system('python scripts/train.py --task pose --variant quantized --data coco8-pose.yaml --imgsz 416 --epochs 200 --seed 42 --warmup 3 --name pose-q-416-seed42')
get_ipython().system('python scripts/train.py --task pose --variant standard --data coco8-pose.yaml --imgsz 416 --epochs 200 --seed 42 --warmup 3 --name pose-std-416-seed42')
print('✅ Pose complete!')

In [ ]:
print('=' * 70)
print('  Multi-Task Results')
print('=' * 70)
for pattern in ['seg-*-416-seed42', 'pose-*-416-seed42']:
    for f in sorted(glob.glob(f'experiments/results/{pattern}/config.json')):
        with open(f) as fh: cfg = json.load(fh)
        name = Path(f).parent.name
        fm = cfg.get('final_metrics', {})
        print(f'\n  {name}: Params={cfg.get("params_M", "?")}M')
        for k, v in fm.items():
            if isinstance(v, (int, float)): print(f'    {k:15s}: {v:.4f}')

## Part C: Quantization

In [ ]:
weights = 'experiments/results/ablation-a4-relu6/best.pt'
if not Path(weights).exists():
    print('Training quick baseline...')
    get_ipython().system('python scripts/train.py --task det --variant quantized --imgsz 416 --epochs 50 --seed 42 --warmup 3 --name quant-baseline')
    weights = 'experiments/results/quant-baseline/best.pt'
print(f'Using: {weights}')
get_ipython().system(f'python scripts/quantize.py --mode ptq --weights {weights} --task det --variant quantized --imgsz 416 --n-calib 500 --backend qnnpack')
print('✅ PTQ complete!')

In [ ]:
get_ipython().system(f'python scripts/quantize.py --mode qat --weights {weights} --task det --variant quantized --imgsz 416 --epochs 10 --lr 1e-4 --backend qnnpack')
print('✅ QAT complete!')

In [ ]:
get_ipython().system(f'python scripts/export.py --weights {weights} --task det --variant quantized --imgsz 416 --formats onnx,torchscript')
print('\nModel Sizes:')
for ext in ['*.onnx', '*.torchscript', '*.pt']:
    for f in sorted(Path('experiments').rglob(ext)):
        print(f'  {f.name}: {f.stat().st_size / 1e6:.2f} MB')

In [ ]:
!cd experiments && zip -r /content/multitask_quant_results.zip results/seg-* results/pose-* results/quant-* 2>/dev/null
print('📥 Download: /content/multitask_quant_results.zip')